In [1]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 1: Setup
# Purpose: Initialize paths, device, imports for ablation study
# ============================================================

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from tqdm.auto import tqdm

PROJECT_ROOT = "/mnt/g/banglafake-detection"
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 256

print("Device:", device)

Device: cuda


In [3]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 2: Define Model Architectures (Baseline + Ablations + Full)
# ============================================================

class BanglaBERTOnly(nn.Module):
    def __init__(self, model_name, num_classes=2, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token (ELECTRA has no pooler)
        output = self.dropout(cls_output)
        return self.classifier(output)


class BanglaBERTWithCNN(nn.Module):
    def __init__(self, model_name, cnn_channels=256, kernel_size=3, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.cnn = nn.Conv1d(
            self.bert.config.hidden_size, cnn_channels, kernel_size,
            padding=kernel_size // 2
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(cnn_channels, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        x = x * mask
        x = x.permute(0, 2, 1)
        x = torch.relu(self.cnn(x))
        pooled = torch.mean(x, dim=2)
        return self.classifier(self.dropout(pooled))


class BanglaBERTWithAttention(nn.Module):
    def __init__(self, model_name, attention_dim=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.attention_projection = nn.Linear(self.bert.config.hidden_size, attention_dim)
        self.attention_score = nn.Linear(attention_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        att_hidden = torch.tanh(self.attention_projection(x))
        att_logits = self.attention_score(att_hidden).squeeze(-1)
        att_logits = att_logits.masked_fill(attention_mask == 0, -1e9)
        att_weights = torch.softmax(att_logits, dim=1)
        attended = (x * att_weights.unsqueeze(-1)).sum(dim=1)
        return self.classifier(self.dropout(attended))


class BanglaBERTFullModel(nn.Module):
    def __init__(self, model_name, cnn_channels=256, kernel_size=3, attention_dim=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.cnn = nn.Conv1d(
            self.bert.config.hidden_size, cnn_channels, kernel_size,
            padding=kernel_size // 2
        )
        self.attention_projection = nn.Linear(cnn_channels, attention_dim)
        self.attention_score = nn.Linear(attention_dim, 1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(cnn_channels, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        x = x * mask
        x = x.permute(0, 2, 1)
        x = torch.relu(self.cnn(x))
        x = x.permute(0, 2, 1)
        att_hidden = torch.tanh(self.attention_projection(x))
        att_logits = self.attention_score(att_hidden).squeeze(-1)
        att_logits = att_logits.masked_fill(attention_mask == 0, -1e9)
        att_weights = torch.softmax(att_logits, dim=1)
        attended = (x * att_weights.unsqueeze(-1)).sum(dim=1)
        return self.classifier(attended)


print("Architectures defined: BanglaBERTOnly, BanglaBERTWithCNN, BanglaBERTWithAttention, BanglaBERTFullModel")

Architectures defined: BanglaBERTOnly, BanglaBERTWithCNN, BanglaBERTWithAttention, BanglaBERTFullModel


In [5]:
# ============================================================
# Cell 3: Load Data, Tokenizer, Dataset, DataLoaders
# ============================================================

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class BanglaFakeNewsDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = str(self.dataframe.iloc[idx]["text"])
        label = int(self.dataframe.iloc[idx]["label"])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }


train_loader = DataLoader(
    BanglaFakeNewsDataset(train_df, tokenizer, MAX_LENGTH),
    batch_size=8, shuffle=True, pin_memory=True
)
val_loader = DataLoader(
    BanglaFakeNewsDataset(val_df, tokenizer, MAX_LENGTH),
    batch_size=8, shuffle=False, pin_memory=True
)
test_loader = DataLoader(
    BanglaFakeNewsDataset(test_df, tokenizer, MAX_LENGTH),
    batch_size=8, shuffle=False, pin_memory=True
)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Train: 2790 Val: 598 Test: 599
